# **Pneumonia Chest X-Ray Classification — Flask Web Application**

This notebook builds a simple **Flask** web front-end around the CNN model trained in `PRAICP-1012-Pneumonia_chest_x-ray_classification.ipynb`. A user can upload a chest X-ray image through a browser page and instantly see whether it is predicted **NORMAL** or **PNEUMONIA**, along with the model's confidence.

## **1. Overview**

**What this notebook does:**
1. Loads the best trained model (`.h5` file) saved by the training notebook.
2. Defines an image preprocessing function that matches the training pipeline (resize, rescale).
3. Builds a single-file Flask application — an HTML upload form plus a `/predict` route — using `render_template_string`, so **no separate `templates/` folder is needed**.
4. Shows two ways to run the app:
   - **As a standalone script** (`python app.py`) — recommended for a real deployment.
   - **Directly inside this notebook** (Jupyter/Colab), by running Flask in a background thread so the notebook kernel is not blocked, optionally tunnelled with `pyngrok` so it is reachable from a public URL when running on Colab.
5. Also exposes a JSON `/api/predict` endpoint so the model can be called programmatically (e.g. from `curl`, Postman, or another application).

## **2. Install & Import Libraries**

Flask serves the web pages/API. `pyngrok` is optional and only needed if you want a public URL while running this notebook on Google Colab (Colab does not expose `localhost` to the outside world by itself).

In [ ]:
# Uncomment the line below the first time you run this in a fresh environment (e.g. Google Colab)
# !pip install flask pyngrok --quiet

import os
import io
import base64
import threading
import numpy as np
from PIL import Image

import tensorflow as tf
from tensorflow.keras.models import load_model

from flask import Flask, request, jsonify, render_template_string
from werkzeug.utils import secure_filename

print('TensorFlow version:', tf.__version__)

## **3. Load the Trained Model**

Point `MODEL_PATH` at whichever `.h5` file the training notebook saved (typically the fine-tuned transfer-learning model, since it had the highest Recall in the comparison table). The cell below will automatically pick the first file it finds among the common checkpoint names if you don't want to set the path manually.

In [ ]:
IMG_SIZE = (150, 150)
CLASS_NAMES = ['NORMAL', 'PNEUMONIA']

# Common checkpoint names produced by the training notebook, in order of preference
CANDIDATE_MODEL_PATHS = [
    'best_fine_tuned.h5',
    'best_mobilenetv2.h5',
    'best_vgg16.h5',
    'best_custom_cnn.h5',
]

MODEL_PATH = next((p for p in CANDIDATE_MODEL_PATHS if os.path.exists(p)), None)

if MODEL_PATH is None:
    raise FileNotFoundError(
        'No trained model file (.h5) was found in the current directory. '
        'Run the training notebook first, or set MODEL_PATH manually to point at your saved model, e.g.:\n'
        "    MODEL_PATH = '/path/to/best_fine_tuned.h5'"
    )

print('Loading model from:', MODEL_PATH)
model = load_model(MODEL_PATH)
model.summary()

## **4. Image Preprocessing Function**

This mirrors exactly what the `ImageDataGenerator` did during training/evaluation: resize to `IMG_SIZE`, convert to a 3-channel array, and rescale pixel values to the `[0, 1]` range.

In [ ]:
def preprocess_image(pil_image):
    """Convert an uploaded PIL image into a model-ready batch of shape (1, H, W, 3)."""
    img = pil_image.convert('RGB').resize(IMG_SIZE)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return np.expand_dims(arr, axis=0)


def predict_image(pil_image):
    """Run the model on a single PIL image and return (label, confidence, raw_probability)."""
    batch = preprocess_image(pil_image)
    prob = float(model.predict(batch, verbose=0)[0][0])   # probability of the PNEUMONIA class
    label = CLASS_NAMES[1] if prob >= 0.5 else CLASS_NAMES[0]
    confidence = prob if prob >= 0.5 else 1 - prob
    return label, confidence, prob

## **5. HTML Templates (inline, no `templates/` folder needed)**

To keep this a single self-contained notebook/file, the page markup is stored as Python strings and rendered with `render_template_string`. A small amount of CSS is included so the page looks reasonably presentable out of the box.

In [ ]:
PAGE_TEMPLATE = '''
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Pneumonia Chest X-Ray Classifier</title>
  <style>
    body { font-family: Arial, Helvetica, sans-serif; background: #f4f6f8; margin: 0; padding: 0; }
    .container { max-width: 640px; margin: 40px auto; background: #fff; padding: 32px;
                 border-radius: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.08); }
    h1 { font-size: 22px; color: #1f2937; margin-bottom: 4px; }
    p.subtitle { color: #6b7280; margin-top: 0; }
    .upload-box { border: 2px dashed #9ca3af; border-radius: 8px; padding: 24px; text-align: center; }
    input[type=submit] { background: #2563eb; color: #fff; border: none; padding: 10px 22px;
                          border-radius: 6px; cursor: pointer; font-size: 15px; margin-top: 12px; }
    input[type=submit]:hover { background: #1d4ed8; }
    .result { margin-top: 28px; padding: 18px; border-radius: 8px; }
    .result.normal { background: #ecfdf5; border: 1px solid #10b981; }
    .result.pneumonia { background: #fef2f2; border: 1px solid #ef4444; }
    .result h2 { margin: 0 0 6px 0; }
    .result .confidence { color: #374151; font-size: 14px; }
    img.preview { max-width: 100%; margin-top: 16px; border-radius: 6px; border: 1px solid #e5e7eb; }
    .disclaimer { margin-top: 24px; font-size: 12px; color: #9ca3af; }
  </style>
</head>
<body>
  <div class="container">
    <h1>Pneumonia Chest X-Ray Classifier</h1>
    <p class="subtitle">Upload a chest X-ray image (JPEG/PNG) to get a NORMAL / PNEUMONIA prediction.</p>

    <form method="POST" enctype="multipart/form-data" class="upload-box">
      <input type="file" name="file" accept="image/*" required>
      <br>
      <input type="submit" value="Classify X-Ray">
    </form>

    {% if label %}
    <div class="result {{ 'pneumonia' if label == 'PNEUMONIA' else 'normal' }}">
      <h2>Prediction: {{ label }}</h2>
      <div class="confidence">Confidence: {{ '%.2f' % (confidence * 100) }}%
        &nbsp;|&nbsp; Raw pneumonia-probability score: {{ '%.4f' % prob }}</div>
      {% if image_data %}
      <img class="preview" src="data:image/png;base64,{{ image_data }}" alt="Uploaded X-Ray">
      {% endif %}
    </div>
    {% endif %}

    <p class="disclaimer">This tool is for educational/demonstration purposes only and is not a medical
       device. It must not be used for real clinical diagnosis without professional oversight.</p>
  </div>
</body>
</html>
'''

## **6. Build the Flask Application**

Two routes are defined:
- **`/`** (GET/POST) — serves the HTML upload page and, on POST, renders the prediction back into the same page.
- **`/api/predict`** (POST) — a JSON API endpoint that accepts a file upload and returns `{label, confidence, probability}`, so the model can be called from other programs, `curl`, or Postman without a browser.

In [ ]:
app = Flask(__name__)
app.config['MAX_CONTENT_LENGTH'] = 8 * 1024 * 1024   # 8 MB upload limit

ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg'}

def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS


@app.route('/', methods=['GET', 'POST'])
def index():
    label = confidence = prob = image_data = None

    if request.method == 'POST':
        file = request.files.get('file')
        if file is None or file.filename == '':
            return render_template_string(PAGE_TEMPLATE, label='ERROR: No file selected',
                                           confidence=0, prob=0, image_data=None)
        if not allowed_file(file.filename):
            return render_template_string(PAGE_TEMPLATE, label='ERROR: Unsupported file type',
                                           confidence=0, prob=0, image_data=None)

        filename = secure_filename(file.filename)
        img_bytes = file.read()
        pil_image = Image.open(io.BytesIO(img_bytes))

        label, confidence, prob = predict_image(pil_image)
        image_data = base64.b64encode(img_bytes).decode('utf-8')

    return render_template_string(PAGE_TEMPLATE, label=label, confidence=confidence,
                                   prob=prob, image_data=image_data)


@app.route('/api/predict', methods=['POST'])
def api_predict():
    file = request.files.get('file')
    if file is None or file.filename == '':
        return jsonify({'error': 'No file provided. Send a multipart/form-data request with a "file" field.'}), 400
    if not allowed_file(file.filename):
        return jsonify({'error': 'Unsupported file type. Use png/jpg/jpeg.'}), 400

    pil_image = Image.open(io.BytesIO(file.read()))
    label, confidence, prob = predict_image(pil_image)
    return jsonify({
        'label': label,
        'confidence': round(confidence, 4),
        'pneumonia_probability': round(prob, 4)
    })


@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model_path': MODEL_PATH})

print('Flask app configured with routes: /  /api/predict  /health')

## **7. Running the App**

###**Option A — Run as a standalone script (recommended for real deployment)**

Save this notebook as a script (`File → Download → Download .py` in Jupyter, or `jupyter nbconvert --to script this_notebook.ipynb`), keep the cell below's `if __name__ == '__main__':` guard, and run:

```bash
python app.py
```

Then open **http://127.0.0.1:5000/** in your browser.

###**Option B — Run directly inside this notebook**

Running `app.run()` directly in a notebook cell blocks the kernel forever (Ctrl+C / Interrupt Kernel to stop it). The cell after this one instead starts Flask in a **background thread**, so you can keep using the notebook while the server runs. If you are on **Google Colab**, `pyngrok` is used to create a public URL, since Colab does not expose `localhost` externally.

In [ ]:
if __name__ == '__main__':
    # This block only runs when the notebook is exported/converted to a plain .py script
    # and executed directly with `python app.py`.
    app.run(host='0.0.0.0', port=5000, debug=False)

In [ ]:
# --- Option B: run inside the notebook without blocking the kernel ---

def run_flask_app():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask_app, daemon=True)
flask_thread.start()
print('Flask server started in a background thread on http://127.0.0.1:5000/')

###**Optional: expose a public URL with `pyngrok` (useful on Google Colab)**

Skip this cell if you are running locally and `http://127.0.0.1:5000/` is already reachable in your browser. On Colab, run the cell below to get a public link to the same app (requires a free ngrok account/auth token — see https://dashboard.ngrok.com/get-started/your-authtoken).

In [ ]:
# from pyngrok import ngrok
#
# NGROK_AUTH_TOKEN = 'YOUR_NGROK_AUTH_TOKEN'   # paste your token from the ngrok dashboard
# ngrok.set_auth_token(NGROK_AUTH_TOKEN)
#
# public_url = ngrok.connect(5000)
# print('Public URL:', public_url)

## **8. Testing the API Endpoint**

Once the server is running (Option A or Option B above), the `/api/predict` endpoint can be tested with a simple `requests` call from a *separate* Python session/cell, or from the command line with `curl`:

```bash
curl -X POST -F "file=@/path/to/xray.jpg" http://127.0.0.1:5000/api/predict
```

In [ ]:
# Example of calling the running app's API from Python (run this in a new cell/session
# AFTER the server from Section 7 has started).
import requests

SAMPLE_IMAGE_PATH = 'path/to/an/xray_image.jpg'   # <-- change this to a real test image on disk

if os.path.exists(SAMPLE_IMAGE_PATH):
    with open(SAMPLE_IMAGE_PATH, 'rb') as f:
        response = requests.post('http://127.0.0.1:5000/api/predict', files={'file': f})
    print(response.status_code, response.json())
else:
    print(f"Set SAMPLE_IMAGE_PATH to a real image file to test the API (currently: '{SAMPLE_IMAGE_PATH}').")

## **9. `requirements.txt` (for deployment)**

If you export this app to run outside the notebook (e.g. on a server, Render, Heroku, or Docker), the following packages are needed:

```text
flask
tensorflow
pillow
numpy
requests
pyngrok        # only needed for the optional Colab public-URL step
```

## **10. Summary**

This notebook wraps the CNN trained in the companion training notebook with a lightweight Flask front-end: a browser-based upload form for interactive use, plus a JSON `/api/predict` endpoint for programmatic/integration use. For production use, consider adding: authentication, request logging, input validation/size limits (a basic 8 MB limit is already set), HTTPS termination via a reverse proxy (e.g. Nginx), and swapping Flask's development server for a production WSGI server such as **gunicorn** or **waitress**.